In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path

BASE_DIR = "/content/drive/MyDrive/MMVision"

Path(f"{BASE_DIR}/models/retrieval").mkdir(parents=True, exist_ok=True)

In [3]:
!pip install git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-d3tq4yxi
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-d3tq4yxi
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=5eab22448e6f1e7ac12346e2e3cf42d6a6ad9038ccd940f408dd335b15e55be2
  Stored in directory: /tmp/pip-ephem-wheel-cache-notmvz4g/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [4]:
%%writefile /content/drive/MyDrive/MMVision/models/retrieval/r5.py

import torch
import torch.nn as nn
from torchvision import models
from torchvision import transforms
import torchvision.models as tv_models
import torch.nn.functional as F
import timm
import numpy as np


device = "cuda" if torch.cuda.is_available() else "cpu"
# -----------------------------
# imageEncoder
# -----------------------------
class ViTEncoder(nn.Module):

    def __init__(
        self,
        embed_dim=512,
        freeze_backbone=True,
        dropout=0.15
    ):
        super().__init__()

        self.freeze_backbone = freeze_backbone

        # --------------------------------------------------
        # ViT-B/16
        # Original ImageNet pretrained ViT-B/16
        # --------------------------------------------------

        self.vit = timm.create_model(
            "vit_base_patch16_224",
            pretrained=True,
            num_classes=0
        )

        # --------------------------------------------------
        # Freeze / unfreeze backbone
        # --------------------------------------------------

        for param in self.vit.parameters():
            param.requires_grad = not freeze_backbone

        # --------------------------------------------------
        # Projection Head
        # 768 -> 1024 -> 512
        # --------------------------------------------------

        self.projection = nn.Sequential(

            nn.Linear(768, 1024),

            nn.LayerNorm(1024),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(1024, embed_dim),

            nn.LayerNorm(embed_dim)
        )

        # --------------------------------------------------
        # Initialization
        # --------------------------------------------------

        for module in self.projection:

            if isinstance(module, nn.Linear):

                nn.init.xavier_uniform_(
                    module.weight
                )

                nn.init.zeros_(
                    module.bias
                )

    def forward(self, images):

        # --------------------------------------------------
        # ViT feature extraction
        # --------------------------------------------------

        if self.freeze_backbone:

            with torch.no_grad():

                features = self.vit.forward_features(
                    images
                )

        else:

            features = self.vit.forward_features(
                images
            )

        # --------------------------------------------------
        # ViT output
        #
        # (B, 197, 768)
        #
        # 1 CLS token
        # 196 patch tokens
        # --------------------------------------------------

        cls_token = features[:, 0]

        # --------------------------------------------------
        # Patch tokens
        # --------------------------------------------------

        patch_tokens = features[:, 1:]

        # --------------------------------------------------
        # Mean pooled patch representation
        # --------------------------------------------------

        patch_mean = patch_tokens.mean(
            dim=1
        )

        # --------------------------------------------------
        # Combine CLS + patch information
        # --------------------------------------------------

        features = (
            cls_token +
            patch_mean
        ) / 2.0

        # --------------------------------------------------
        # Projection
        # --------------------------------------------------

        embeddings = self.projection(
            features
        )

        # --------------------------------------------------
        # L2 normalization
        # --------------------------------------------------

        embeddings = F.normalize(
            embeddings,
            p=2,
            dim=1
        )

        return embeddings
# -----------------------------
# textencoder
# -----------------------------
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len).unsqueeze(1).float()

        div_term = torch.exp(
            torch.arange(0, embed_dim, 2).float()
            * (-math.log(10000.0) / embed_dim)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)      # (1, max_len, embed_dim)

        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TextEncoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim=512,
        num_heads=8,
        num_layers=4,
        ff_dim=2048,
        pad_idx=0,
        dropout=0.15
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=pad_idx
        )

        self.position = PositionalEncoding(
            embed_dim
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.projection = nn.Sequential(
            nn.Linear(embed_dim, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(1024, 512),
            nn.LayerNorm(512)
        )

        for module in self.projection:

            if isinstance(module, nn.Linear):

                nn.init.xavier_uniform_(
                    module.weight
                )

                nn.init.zeros_(
                    module.bias
                )

    def forward(
        self,
        captions,
        lengths
    ):

        x = self.embedding(captions)

        x = self.position(x)

        device = captions.device

        max_len = captions.size(1)

        lengths = lengths.to(device)

        mask = (
            torch.arange(
                max_len,
                device=device
            )
            .unsqueeze(0)
            .expand(captions.size(0), -1)
            >= lengths.unsqueeze(1)
        )

        x = self.transformer(
            x,
            src_key_padding_mask=mask
        )

        # Masked mean pooling

        valid_mask = (~mask).unsqueeze(-1).float()

        x = (
            x * valid_mask
        ).sum(dim=1) / valid_mask.sum(
            dim=1
        ).clamp(min=1e-8)

        embeddings = self.projection(x)

        embeddings = F.normalize(
            embeddings,
            p=2,
            dim=1
        )

        return embeddings
# -----------------------------
# Joint Model
# -----------------------------
class ViTTransformerRetrieval(nn.Module):

    def __init__(self, image_encoder, text_encoder):
        super().__init__()

        self.image_encoder = image_encoder
        self.text_encoder = text_encoder

        # Learnable CLIP-style temperature
        self.logit_scale = nn.Parameter(
            torch.tensor(np.log(1 / 0.07), dtype=torch.float32)
        )

    def forward(self, images, captions, lengths):

        image_emb = self.image_encoder(images)

        text_emb = self.text_encoder(
            captions,
            lengths
        )

        return image_emb, text_emb

# -----------------------------
#image transform
# -----------------------------

image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# -----------------------------
# Predict Caption
# -----------------------------
def encode_image(model, image):
    model.eval()
    image = image_transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        embedding = model.image_encoder(image)
    return embedding

def encode_caption(model, caption, vocab):
    model.eval()
    tokens = caption.lower().split()
    ids = [vocab["<SOS>"]]
    for token in tokens:
        ids.append(vocab.get(token, vocab["<UNK>"]))

    ids.append(vocab["<EOS>"])
    caption_tensor = torch.tensor(ids).unsqueeze(0).to(device)
    lengths = torch.tensor([len(ids)])

    with torch.no_grad():
        embedding = model.text_encoder(caption_tensor,lengths)

    return embedding

def image_to_images(model,image,image_db,image_paths,topk=5):
    query = encode_image(model, image)
    sims = torch.mm(query, image_db.T).squeeze(0)
    idx = sims.topk(topk).indices.cpu().tolist()
    return [image_paths[i] for i in idx]

def caption_to_images(model,caption,vocab,image_db,image_paths,topk=5):
    query = encode_caption(model,caption,vocab)
    sims = torch.mm(query, image_db.T).squeeze(0)
    idx = sims.topk(topk).indices.cpu().tolist()
    return [image_paths[i] for i in idx]

def image_to_captions(model,image,caption_db,captions,topk=5):
    query = encode_image(model, image)
    sims = torch.mm(query, caption_db.T).squeeze(0)
    idx = sims.topk(topk).indices.cpu().tolist()
    return [captions[i] for i in idx]

def caption_to_captions(model,caption,vocab,caption_db,captions,topk=5):
    query = encode_caption(model,caption,vocab)
    sims = torch.mm(query, caption_db.T).squeeze(0)
    idx = sims.topk(topk).indices.cpu().tolist()
    return [captions[i] for i in idx]

# -----------------------------
# Build Model
# -----------------------------
def build_R5(vocab):
  image_encoder = ViTEncoder(freeze_backbone=True).to(device)

  text_encoder = TextEncoder(
      vocab_size=len(vocab),
      embed_dim=512,
      pad_idx=vocab["<PAD>"]
  ).to(device)

  model = ViTTransformerRetrieval(
      image_encoder=image_encoder,
      text_encoder=text_encoder
  ).to(device)

  return model

Overwriting /content/drive/MyDrive/MMVision/models/retrieval/r5.py


In [ ]:
%%writefile /content/drive/MyDrive/MMVision/models/retrieval/r3.py
r3
import torch
import torch.nn as nn
from torchvision import models
from torchvision import transforms
import torchvision.models as tv_models
from torch.nn.utils.rnn import pack_padded_sequence
from torch.nn.utils.rnn import pad_packed_sequence
import torch.nn.functional as F
import timm
import math
import numpy as np


device = "cuda" if torch.cuda.is_available() else "cpu"
# -----------------------------
# imageEncoder
# -----------------------------
class ImageEncoder(nn.Module):

    def __init__(self, embed_dim=768):

        super().__init__()

        backbone = models.resnet50(
            weights=models.ResNet50_Weights.IMAGENET1K_V2
        )

        for param in backbone.parameters():
          param.requires_grad = False

        # Fine tune last two ResNet stages
        for param in backbone.layer3.parameters():
            param.requires_grad = True

        for param in backbone.layer4.parameters():
            param.requires_grad = True

        self.backbone = nn.Sequential(
            *list(backbone.children())[:-1]
        )

        self.projection = nn.Sequential(
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(1024, embed_dim)
        )

    def forward(self, images):

        features = self.backbone(images)

        features = features.squeeze(-1).squeeze(-1)

        embeddings = self.projection(features)

        embeddings = F.normalize(
            embeddings,
            p=2,
            dim=1
        )

        return embeddings
# -----------------------------
# textencoder
# -----------------------------

class TextEncoderAttention(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim=300,
        hidden_dim=512,
        pad_idx=0
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=pad_idx
        )

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=2,
            dropout=0.5,
            bidirectional=True,
            batch_first=True
        )

        # Attention layer
        self.attn_W = nn.Linear(hidden_dim * 2, hidden_dim)
        self.attn_U = nn.Linear(hidden_dim * 2, hidden_dim)
        self.attn_v = nn.Linear(hidden_dim, 1, bias=False)

        self.projection = nn.Sequential(
            nn.Linear(hidden_dim * 2, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 768)
        )

    def forward(
        self,
        captions,
        lengths
    ):

        embedded = self.embedding(captions)

        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_out, (hidden, cell) = self.lstm(packed)

        outputs, _ = pad_packed_sequence(
            packed_out,
            batch_first=True
        )

        # outputs:
        # (batch, seq_len, hidden_dim*2)
        batch_size = outputs.size(0)
        seq_len = outputs.size(1)

        # Final BiLSTM hidden state (query)
        hidden_forward = hidden[-2]
        hidden_backward = hidden[-1]

        query = torch.cat(
            [hidden_forward, hidden_backward],
            dim=1
        )

        query = query.unsqueeze(1).repeat(
            1,
            seq_len,
            1
        )

        # Bahdanau Attention
        energy = torch.tanh(
            self.attn_W(outputs)
            +
            self.attn_U(query)
        )

        attn_scores = self.attn_v(energy).squeeze(-1)

        mask = (
            torch.arange(
                seq_len,
                device=outputs.device
            )
            .expand(batch_size, seq_len)
            >= lengths.unsqueeze(1).to(outputs.device)
        )

        attn_scores = attn_scores.masked_fill(
            mask,
            -1e9
        )

        attn_weights = torch.softmax(
            attn_scores,
            dim=1
        )

        context = torch.sum(
            outputs *
            attn_weights.unsqueeze(-1),
            dim=1
        )



        embeddings = self.projection(context)

        embeddings = F.normalize(
            embeddings,
            p=2,
            dim=1
        )

        return embeddings
# -----------------------------
# Joint Model
# -----------------------------

class ResNetLSTMRetrieval(nn.Module):
    def __init__(self, image_encoder, text_encoder):
        super().__init__()

        self.image_encoder = image_encoder
        self.text_encoder = text_encoder

        # Learnable temperature parameter
        self.logit_scale = nn.Parameter(
            torch.ones([]) * np.log(1 / 0.07)
        )

    def forward(self, images, captions, lengths):
        image_emb = self.image_encoder(images)
        text_emb = self.text_encoder(captions, lengths)

        # Normalize embeddings
        image_emb = F.normalize(image_emb, p=2, dim=1)
        text_emb = F.normalize(text_emb, p=2, dim=1)

        return image_emb, text_emb


# -----------------------------
#image transform
# -----------------------------

image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
# -----------------------------
# Predict Caption
# -----------------------------
def encode_image(model, image):
    model.eval()
    image = image_transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        embedding = model.image_encoder(image)
    return embedding

def encode_caption(model, caption, vocab):
    model.eval()
    tokens = caption.lower().split()
    ids = [vocab["<SOS>"]]
    for token in tokens:
        ids.append(vocab.get(token, vocab["<UNK>"]))

    ids.append(vocab["<EOS>"])
    caption_tensor = torch.tensor(ids).unsqueeze(0).to(device)
    lengths = torch.tensor([len(ids)])

    with torch.no_grad():
        embedding = model.text_encoder(caption_tensor,lengths)

    return embedding

def image_to_images(model,image,image_db,image_paths,topk=5):
    query = encode_image(model, image)
    sims = torch.mm(query, image_db.T).squeeze(0)
    idx = sims.topk(topk).indices.cpu().tolist()
    return [image_paths[i] for i in idx]

def caption_to_images(model,caption,vocab,image_db,image_paths,topk=5):
    query = encode_caption(model,caption,vocab)
    sims = torch.mm(query, image_db.T).squeeze(0)
    idx = sims.topk(topk).indices.cpu().tolist()
    return [image_paths[i] for i in idx]

def image_to_captions(model,image,caption_db,captions,topk=5):
    query = encode_image(model, image)
    sims = torch.mm(query, caption_db.T).squeeze(0)
    idx = sims.topk(topk).indices.cpu().tolist()
    return [captions[i] for i in idx]

def caption_to_captions(model,caption,vocab,caption_db,captions,topk=5):
    query = encode_caption(model,caption,vocab)
    sims = torch.mm(query, caption_db.T).squeeze(0)
    idx = sims.topk(topk).indices.cpu().tolist()
    return [captions[i] for i in idx]

# -----------------------------
# Build Model
# -----------------------------
def build_R3(vocab):
  encoder = ImageEncoder(embed_dim=768).to(device)

  text_encoder = TextEncoderAttention(vocab_size=len(vocab),embed_dim=300,hidden_dim=512,pad_idx=vocab["<PAD>"]).to(device)

  model = ResNetLSTMRetrieval(image_encoder=encoder,text_encoder=text_encoder).to(device)

  return model

Overwriting /content/drive/MyDrive/MMVision/models/retrieval/r3.py
